In [1]:
import sys
from pathlib import Path

print("Python đang sử dụng:")
print(sys.executable)

print("\nThư mục notebook:")
print(Path.cwd())

Python đang sử dụng:
C:\Volume D\Hệ gợi ý\Jupyter\Project\.venv\Scripts\python.exe

Thư mục notebook:
C:\Volume D\Hệ gợi ý\Jupyter\Project\notebooks


In [2]:
from pathlib import Path
import pandas as pd
import numpy as np
import json
import gc

# Notebook hiện nằm trong Project/notebooks
# Vì vậy thư mục Project là thư mục cha
CURRENT_DIR = Path.cwd()

if CURRENT_DIR.name == "notebooks":
    ROOT = CURRENT_DIR.parent
else:
    ROOT = CURRENT_DIR

RAW_DIR = ROOT / "data" / "raw"
INTERIM_DIR = ROOT / "data" / "interim"
PROCESSED_DIR = ROOT / "data" / "processed"
FEATURE_DIR = ROOT / "data" / "features"
SPLIT_DIR = ROOT / "data" / "splits"
REPORT_DIR = ROOT / "reports"

for folder in [
    RAW_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    FEATURE_DIR,
    SPLIT_DIR,
    REPORT_DIR
]:
    folder.mkdir(parents=True, exist_ok=True)

print("Thư mục dự án:", ROOT)
print("Thư mục dữ liệu raw:", RAW_DIR)

Thư mục dự án: C:\Volume D\Hệ gợi ý\Jupyter\Project
Thư mục dữ liệu raw: C:\Volume D\Hệ gợi ý\Jupyter\Project\data\raw


In [3]:
required_files = [
    "events.csv",
    "item_properties_part1.csv",
    "item_properties_part2.csv",
    "category_tree.csv"
]

for filename in required_files:
    path = RAW_DIR / filename
    status = "Đã có" if path.exists() else "THIẾU"
    print(f"{status:8} | {filename}")

Đã có    | events.csv
Đã có    | item_properties_part1.csv
Đã có    | item_properties_part2.csv
Đã có    | category_tree.csv


In [4]:
event_dtypes = {
    "timestamp": "int64",
    "visitorid": "int64",
    "event": "string",
    "itemid": "int64",
    "transactionid": "Int64"
}

events_raw = pd.read_csv(
    RAW_DIR / "events.csv",
    dtype=event_dtypes
)

print("Kích thước dữ liệu:", events_raw.shape)
events_raw.head()

Kích thước dữ liệu: (2756101, 5)


,timestamp,visitorid,event,itemid,transactionid
0,1433221332117,257597,view,355908,<NA>
1,1433224214164,992329,view,248676,<NA>
2,1433221999827,111016,view,318965,<NA>
3,1433221955914,483717,view,253185,<NA>
4,1433221337106,951259,view,367447,<NA>


In [5]:
print("Thông tin dữ liệu:")
events_raw.info()

print("\nSố lượng theo hành vi:")
print(events_raw["event"].value_counts(dropna=False))

print("\nSố giá trị thiếu:")
print(events_raw.isna().sum())

print("\nSố người dùng:")
print(events_raw["visitorid"].nunique())

print("\nSố sản phẩm:")
print(events_raw["itemid"].nunique())

print("\nSố dòng trùng hoàn toàn:")
print(events_raw.duplicated().sum())

Thông tin dữ liệu:
<class 'pandas.DataFrame'>
RangeIndex: 2756101 entries, 0 to 2756100
Data columns (total 5 columns):
 #   Column         Dtype 
---  ------         ----- 
 0   timestamp      int64 
 1   visitorid      int64 
 2   event          string
 3   itemid         int64 
 4   transactionid  Int64 
dtypes: Int64(1), int64(3), string(1)
memory usage: 118.8 MB

Số lượng theo hành vi:
event
view           2664312
addtocart        69332
transaction      22457
Name: count, dtype: int64[pyarrow]

Số giá trị thiếu:
timestamp              0
visitorid              0
event                  0
itemid                 0
transactionid    2733644
dtype: int64

Số người dùng:
1407580

Số sản phẩm:
235061

Số dòng trùng hoàn toàn:
460


In [6]:
events = events_raw.copy()

valid_event_types = ["view", "addtocart", "transaction"]

# Chỉ giữ các hành vi hợp lệ
events = events[
    events["event"].isin(valid_event_types)
].copy()

# Xóa dòng thiếu các trường bắt buộc
events = events.dropna(
    subset=["timestamp", "visitorid", "event", "itemid"]
)

# Xóa dòng trùng hoàn toàn
before = len(events)
events = events.drop_duplicates()
after = len(events)

print("Số dòng trước khi xóa trùng:", before)
print("Số dòng sau khi xóa trùng:", after)
print("Số dòng đã xóa:", before - after)

# Chuyển timestamp millisecond sang thời gian đọc được
events["event_time"] = pd.to_datetime(
    events["timestamp"],
    unit="ms",
    utc=True
)

# Sắp xếp theo người dùng và thời gian
events = events.sort_values(
    ["visitorid", "event_time", "itemid"]
).reset_index(drop=True)

events.head()

Số dòng trước khi xóa trùng: 2756101
Số dòng sau khi xóa trùng: 2755641
Số dòng đã xóa: 460


,timestamp,visitorid,event,itemid,transactionid,event_time
0,1442004589439,0,view,285930,<NA>,2015-09-11 20:49:49.439000+00:00
1,1442004759591,0,view,357564,<NA>,2015-09-11 20:52:39.591000+00:00
2,1442004917175,0,view,67045,<NA>,2015-09-11 20:55:17.175000+00:00
3,1439487966444,1,view,72028,<NA>,2015-08-13 17:46:06.444000+00:00
4,1438969904567,2,view,325215,<NA>,2015-08-07 17:51:44.567000+00:00


In [7]:
print("Thời điểm đầu:", events["event_time"].min())
print("Thời điểm cuối:", events["event_time"].max())

print("\nPhân bố hành vi:")
print(events["event"].value_counts())

print("\nSố người dùng:", events["visitorid"].nunique())
print("Số sản phẩm:", events["itemid"].nunique())
print("Số sự kiện:", len(events))

Thời điểm đầu: 2015-05-03 03:00:04.384000+00:00
Thời điểm cuối: 2015-09-18 02:59:47.788000+00:00

Phân bố hành vi:
event
view           2664218
addtocart        68966
transaction      22457
Name: count, dtype: int64[pyarrow]

Số người dùng: 1407580
Số sản phẩm: 235061
Số sự kiện: 2755641


In [8]:
# Khoảng cách thời gian so với hành vi trước của cùng người dùng
time_gap = (
    events.groupby("visitorid")["event_time"]
    .diff()
)

# Người dùng thay đổi hoặc khoảng nghỉ lớn hơn 30 phút
new_user = events["visitorid"].ne(events["visitorid"].shift())

new_session = (
    new_user
    | time_gap.isna()
    | time_gap.gt(pd.Timedelta(minutes=30))
)

# ID session duy nhất trên toàn bộ dữ liệu
events["session_id"] = new_session.cumsum().astype("int64")

# Vị trí hành vi trong session: 0, 1, 2, ...
events["session_position"] = (
    events.groupby("session_id")
    .cumcount()
    .astype("int32")
)

# Số sự kiện trong session
events["session_length"] = (
    events.groupby("session_id")["itemid"]
    .transform("size")
    .astype("int32")
)

print("Số session:", events["session_id"].nunique())
events[
    [
        "visitorid",
        "event_time",
        "event",
        "itemid",
        "session_id",
        "session_position",
        "session_length"
    ]
].head(20)

Số session: 1761675


,visitorid,event_time,event,itemid,session_id,session_position,session_length
0,0,2015-09-11 20:49:49.439000+00:00,view,285930,1,0,3
1,0,2015-09-11 20:52:39.591000+00:00,view,357564,1,1,3
2,0,2015-09-11 20:55:17.175000+00:00,view,67045,1,2,3
3,1,2015-08-13 17:46:06.444000+00:00,view,72028,2,0,1
4,2,2015-08-07 17:51:44.567000+00:00,view,325215,3,0,8
5,2,2015-08-07 17:53:33.790000+00:00,view,325215,3,1,8
6,2,2015-08-07 17:56:52.664000+00:00,view,259884,3,2,8
7,2,2015-08-07 18:01:08.920000+00:00,view,216305,3,3,8
8,2,2015-08-07 18:08:25.669000+00:00,view,342816,3,4,8
9,2,2015-08-07 18:17:24.375000+00:00,view,342816,3,5,8


In [9]:
# Hai mốc thời gian 80% và 90%
train_cutoff = events["event_time"].quantile(0.80)
validation_cutoff = events["event_time"].quantile(0.90)

print("Train cutoff:", train_cutoff)
print("Validation cutoff:", validation_cutoff)

# Thời gian bắt đầu và kết thúc của từng session
session_bounds = (
    events.groupby("session_id")
    .agg(
        session_start=("event_time", "min"),
        session_end=("event_time", "max"),
        visitorid=("visitorid", "first"),
        number_of_events=("itemid", "size")
    )
    .reset_index()
)

def assign_split(row):
    start = row["session_start"]
    end = row["session_end"]

    if end <= train_cutoff:
        return "train"

    if start > train_cutoff and end <= validation_cutoff:
        return "validation"

    if start > validation_cutoff:
        return "test"

    # Session đi qua biên chia dữ liệu
    return "boundary_drop"

session_bounds["split"] = session_bounds.apply(
    assign_split,
    axis=1
)

print(session_bounds["split"].value_counts())

Train cutoff: 2015-08-18 04:23:20.445000+00:00
Validation cutoff: 2015-09-02 17:49:14.032000+00:00
split
train            1396273
test              182821
validation        182534
boundary_drop         47
Name: count, dtype: int64


In [10]:
events = events.merge(
    session_bounds[["session_id", "split"]],
    on="session_id",
    how="left",
    validate="many_to_one"
)

print(events["split"].value_counts())

split
train            2204470
test              275484
validation        275359
boundary_drop        328
Name: count, dtype: int64


In [11]:
train_events = events[events["split"] == "train"].copy()

train_users = set(train_events["visitorid"].unique())
train_items = set(train_events["itemid"].unique())

events["user_seen_in_train"] = events["visitorid"].isin(train_users)
events["item_seen_in_train"] = events["itemid"].isin(train_items)

conditions = [
    (~events["user_seen_in_train"]) & (~events["item_seen_in_train"]),
    (~events["user_seen_in_train"]) & (events["item_seen_in_train"]),
    (events["user_seen_in_train"]) & (~events["item_seen_in_train"])
]

values = [
    "cold_both",
    "cold_user",
    "cold_item"
]

events["cold_start_type"] = np.select(
    conditions,
    values,
    default="warm"
)

print(
    events[events["split"].isin(["validation", "test"])]
    ["cold_start_type"]
    .value_counts()
)

cold_start_type
cold_user    439501
warm          71154
cold_both     33691
cold_item      6497
Name: count, dtype: int64


In [12]:
events_output = PROCESSED_DIR / "events_clean_split.parquet"

events.to_parquet(
    events_output,
    index=False
)

print("Đã lưu:", events_output)
print("Dung lượng MB:", round(events_output.stat().st_size / 1024**2, 2))

Đã lưu: C:\Volume D\Hệ gợi ý\Jupyter\Project\data\processed\events_clean_split.parquet
Dung lượng MB: 62.88


In [13]:
PROPERTY_CHUNK_DIR = INTERIM_DIR / "property_chunks"
PROPERTY_CHUNK_DIR.mkdir(parents=True, exist_ok=True)

property_files = [
    RAW_DIR / "item_properties_part1.csv",
    RAW_DIR / "item_properties_part2.csv"
]

relevant_properties = ["categoryid", "available"]
chunksize = 1_000_000

for part_number, file_path in enumerate(property_files, start=1):
    print("\nĐang xử lý:", file_path.name)

    reader = pd.read_csv(
        file_path,
        usecols=["timestamp", "itemid", "property", "value"],
        dtype={
            "timestamp": "int64",
            "itemid": "int64",
            "property": "string",
            "value": "string"
        },
        chunksize=chunksize
    )

    for chunk_number, chunk in enumerate(reader):
        filtered = chunk[
            chunk["property"].isin(relevant_properties)
        ].copy()

        filtered = filtered.dropna(
            subset=["timestamp", "itemid", "property", "value"]
        )

        filtered = filtered.drop_duplicates()

        filtered["property_time"] = pd.to_datetime(
            filtered["timestamp"],
            unit="ms",
            utc=True
        )

        output_file = (
            PROPERTY_CHUNK_DIR
            / f"properties_part{part_number}_{chunk_number:03d}.parquet"
        )

        filtered.to_parquet(output_file, index=False)

        print(
            f"  Chunk {chunk_number:02d}: "
            f"{len(chunk):,} raw → {len(filtered):,} được giữ"
        )

        del chunk, filtered
        gc.collect()

print("\nĐã xử lý xong hai file thuộc tính.")


Đang xử lý: item_properties_part1.csv
  Chunk 00: 1,000,000 raw → 112,246 được giữ
  Chunk 01: 1,000,000 raw → 111,802 được giữ
  Chunk 02: 1,000,000 raw → 112,474 được giữ
  Chunk 03: 1,000,000 raw → 113,205 được giữ
  Chunk 04: 1,000,000 raw → 113,719 được giữ
  Chunk 05: 1,000,000 raw → 113,592 được giữ
  Chunk 06: 1,000,000 raw → 113,096 được giữ
  Chunk 07: 1,000,000 raw → 113,435 được giữ
  Chunk 08: 1,000,000 raw → 113,397 được giữ
  Chunk 09: 1,000,000 raw → 113,111 được giữ
  Chunk 10: 999,999 raw → 113,615 được giữ

Đang xử lý: item_properties_part2.csv
  Chunk 00: 1,000,000 raw → 112,721 được giữ
  Chunk 01: 1,000,000 raw → 112,979 được giữ
  Chunk 02: 1,000,000 raw → 112,845 được giữ
  Chunk 03: 1,000,000 raw → 113,087 được giữ
  Chunk 04: 1,000,000 raw → 113,450 được giữ
  Chunk 05: 1,000,000 raw → 113,287 được giữ
  Chunk 06: 1,000,000 raw → 113,294 được giữ
  Chunk 07: 1,000,000 raw → 113,118 được giữ
  Chunk 08: 1,000,000 raw → 112,133 được giữ
  Chunk 09: 275,903 raw 

In [14]:
property_chunk_files = sorted(
    PROPERTY_CHUNK_DIR.glob("properties_part*.parquet")
)

print("Số file trung gian:", len(property_chunk_files))

if len(property_chunk_files) == 0:
    raise FileNotFoundError(
        "Không tìm thấy file thuộc tính trung gian. "
        "Hãy chạy lại ô xử lý item properties."
    )

properties = pd.concat(
    [pd.read_parquet(path) for path in property_chunk_files],
    ignore_index=True
)

properties = properties.drop_duplicates().reset_index(drop=True)

print("Số dòng thuộc tính category/available:", len(properties))
print(properties["property"].value_counts())

properties.head()

Số file trung gian: 21
Số dòng thuộc tính category/available: 2291853
property
available     1503639
categoryid     788214
Name: count, dtype: int64[pyarrow]


,timestamp,itemid,property,value,property_time
0,1435460400000,460429,categoryid,1338,2015-06-28 03:00:00+00:00
1,1436065200000,285026,available,0,2015-07-05 03:00:00+00:00
2,1437274800000,186518,available,0,2015-07-19 03:00:00+00:00
3,1433646000000,423682,available,0,2015-06-07 03:00:00+00:00
4,1434250800000,316253,available,1,2015-06-14 03:00:00+00:00


In [16]:
properties_output = INTERIM_DIR / "item_properties_relevant.parquet"

properties.to_parquet(
    properties_output,
    index=False
)

print("Đã lưu:", properties_output)

Đã lưu: C:\Volume D\Hệ gợi ý\Jupyter\Project\data\interim\item_properties_relevant.parquet


In [17]:
properties_train = properties[
    properties["property_time"] <= train_cutoff
].copy()

# Sắp xếp để dòng cuối là trạng thái mới nhất
properties_train = properties_train.sort_values(
    ["itemid", "property", "property_time"]
)

latest_properties = properties_train.drop_duplicates(
    subset=["itemid", "property"],
    keep="last"
)

item_snapshot = latest_properties.pivot(
    index="itemid",
    columns="property",
    values="value"
).reset_index()

item_snapshot.columns.name = None

# Đổi tên rõ nghĩa hơn
if "categoryid" in item_snapshot.columns:
    item_snapshot = item_snapshot.rename(
        columns={"categoryid": "category_id"}
    )
    item_snapshot["category_id"] = pd.to_numeric(
        item_snapshot["category_id"],
        errors="coerce"
    ).astype("Int64")

if "available" in item_snapshot.columns:
    item_snapshot["available"] = pd.to_numeric(
        item_snapshot["available"],
        errors="coerce"
    ).astype("Int8")

print("Số sản phẩm có metadata trước train cutoff:", len(item_snapshot))
item_snapshot.head()

Số sản phẩm có metadata trước train cutoff: 412882


,itemid,available,category_id
0,0,0,209
1,1,0,1114
2,2,0,1305
3,3,0,1171
4,4,0,1038


In [18]:
category_tree = pd.read_csv(
    RAW_DIR / "category_tree.csv",
    dtype={
        "categoryid": "Int64",
        "parentid": "Int64"
    }
)

category_tree = (
    category_tree
    .dropna(subset=["categoryid"])
    .drop_duplicates(subset=["categoryid"])
    .reset_index(drop=True)
)

print("Số danh mục:", len(category_tree))
category_tree.head()

Số danh mục: 1669


,categoryid,parentid
0,1016,213
1,809,169
2,570,9
3,1691,885
4,536,1691


In [19]:
parent_map = dict(
    zip(
        category_tree["categoryid"],
        category_tree["parentid"]
    )
)

def get_category_information(category_id):
    current = int(category_id)
    leaf_to_root = []
    visited = set()

    while current not in visited:
        visited.add(current)
        leaf_to_root.append(current)

        parent = parent_map.get(current, pd.NA)

        if pd.isna(parent):
            break

        current = int(parent)

    root_to_leaf = list(reversed(leaf_to_root))

    return (
        root_to_leaf[0],                 # root category
        len(root_to_leaf) - 1,           # depth
        "/".join(map(str, root_to_leaf)) # category path
    )

category_information = category_tree["categoryid"].apply(
    get_category_information
)

category_tree[
    ["root_categoryid", "category_depth", "category_path"]
] = pd.DataFrame(
    category_information.tolist(),
    index=category_tree.index
)

category_tree["root_categoryid"] = (
    category_tree["root_categoryid"].astype("Int64")
)

category_tree.head()

,categoryid,parentid,root_categoryid,category_depth,category_path
0,1016,213,1532,3,1532/1299/213/1016
1,809,169,395,3,395/1257/169/809
2,570,9,653,3,653/351/9/570
3,1691,885,1579,2,1579/885/1691
4,536,1691,1579,3,1579/885/1691/536


In [20]:
category_output = PROCESSED_DIR / "category_tree_clean.parquet"

category_tree.to_parquet(
    category_output,
    index=False
)

print("Đã lưu:", category_output)

Đã lưu: C:\Volume D\Hệ gợi ý\Jupyter\Project\data\processed\category_tree_clean.parquet


In [21]:
train_item_ids = pd.Index(train_events["itemid"].unique())
metadata_item_ids = pd.Index(item_snapshot["itemid"].unique())

all_known_items = train_item_ids.union(metadata_item_ids)

products = pd.DataFrame({
    "itemid": all_known_items.astype("int64")
})

products = products.merge(
    item_snapshot,
    on="itemid",
    how="left"
)

products["seen_in_train"] = products["itemid"].isin(train_items)

products = products.merge(
    category_tree[
        [
            "categoryid",
            "parentid",
            "root_categoryid",
            "category_depth",
            "category_path"
        ]
    ],
    left_on="category_id",
    right_on="categoryid",
    how="left"
)

products = products.drop(
    columns=["categoryid"],
    errors="ignore"
)

# Chỉ loại khi biết chắc available = 0.
# Nếu thiếu trạng thái, vẫn giữ làm ứng viên.
products["candidate_at_train"] = (
    products["available"].isna()
    | products["available"].eq(1)
)

print("Tổng số sản phẩm:", len(products))
print("Sản phẩm đã xuất hiện trong train:", products["seen_in_train"].sum())
print(
    "Sản phẩm có metadata nhưng chưa có tương tác train:",
    (~products["seen_in_train"]).sum()
)

products.head()

Tổng số sản phẩm: 456974
Sản phẩm đã xuất hiện trong train: 212915
Sản phẩm có metadata nhưng chưa có tương tác train: 244059


,itemid,available,category_id,seen_in_train,parentid,root_categoryid,category_depth,category_path,candidate_at_train
0,0,0,209,False,293,1532,2.0,1532/293/209,False
1,1,0,1114,False,113,1532,2.0,1532/113/1114,False
2,2,0,1305,False,1214,653,3.0,653/1202/1214/1305,False
3,3,0,1171,False,938,1532,3.0,1532/986/938/1171,False
4,4,0,1038,True,1174,1532,3.0,1532/107/1174/1038,False


In [22]:
products_output = PROCESSED_DIR / "products_train_snapshot.parquet"

products.to_parquet(
    products_output,
    index=False
)

print("Đã lưu:", products_output)

Đã lưu: C:\Volume D\Hệ gợi ý\Jupyter\Project\data\processed\products_train_snapshot.parquet


In [23]:
interaction_counts = (
    train_events.groupby(
        ["visitorid", "itemid", "event"]
    )
    .size()
    .unstack(fill_value=0)
    .reset_index()
)

# Đảm bảo luôn có đủ ba cột
for column in ["view", "addtocart", "transaction"]:
    if column not in interaction_counts.columns:
        interaction_counts[column] = 0

for column in ["view", "addtocart", "transaction"]:
    interaction_counts[column] = (
        interaction_counts[column].astype("int32")
    )

interaction_counts["interaction_score"] = (
    np.log1p(interaction_counts["view"])
    + 3 * np.log1p(interaction_counts["addtocart"])
    + 5 * np.log1p(interaction_counts["transaction"])
)

interaction_counts["has_cart"] = (
    interaction_counts["addtocart"] > 0
)

interaction_counts["has_purchase"] = (
    interaction_counts["transaction"] > 0
)

last_interaction = (
    train_events.groupby(["visitorid", "itemid"])
    ["event_time"]
    .max()
    .rename("last_interaction_time")
    .reset_index()
)

interactions_train = interaction_counts.merge(
    last_interaction,
    on=["visitorid", "itemid"],
    how="left",
    validate="one_to_one"
)

print("Số cặp user-item:", len(interactions_train))
interactions_train.head()

Số cặp user-item: 1713144


,visitorid,itemid,addtocart,transaction,view,interaction_score,has_cart,has_purchase,last_interaction_time
0,1,72028,0,0,1,0.693147,False,False,2015-08-13 17:46:06.444000+00:00
1,2,216305,0,0,2,1.098612,False,False,2015-08-07 18:17:43.170000+00:00
2,2,259884,0,0,1,0.693147,False,False,2015-08-07 17:56:52.664000+00:00
3,2,325215,0,0,3,1.386294,False,False,2015-08-07 18:20:57.845000+00:00
4,2,342816,0,0,2,1.098612,False,False,2015-08-07 18:17:24.375000+00:00


In [24]:
interaction_output = PROCESSED_DIR / "interactions_train.parquet"

interactions_train.to_parquet(
    interaction_output,
    index=False
)

print("Đã lưu:", interaction_output)

Đã lưu: C:\Volume D\Hệ gợi ý\Jupyter\Project\data\processed\interactions_train.parquet


In [25]:
user_mapping = pd.DataFrame({
    "visitorid": np.sort(train_events["visitorid"].unique())
})

user_mapping["user_idx"] = np.arange(
    len(user_mapping),
    dtype=np.int32
)

item_mapping = pd.DataFrame({
    "itemid": np.sort(train_events["itemid"].unique())
})

item_mapping["item_idx"] = np.arange(
    len(item_mapping),
    dtype=np.int32
)

encoded_interactions = (
    interactions_train
    .merge(
        user_mapping,
        on="visitorid",
        how="inner",
        validate="many_to_one"
    )
    .merge(
        item_mapping,
        on="itemid",
        how="inner",
        validate="many_to_one"
    )
)

print("Số user được mã hóa:", len(user_mapping))
print("Số item được mã hóa:", len(item_mapping))
encoded_interactions.head()

Số user được mã hóa: 1123755
Số item được mã hóa: 212915


,visitorid,itemid,addtocart,transaction,view,interaction_score,has_cart,has_purchase,last_interaction_time,user_idx,item_idx
0,1,72028,0,0,1,0.693147,False,False,2015-08-13 17:46:06.444000+00:00,0,32979
1,2,216305,0,0,2,1.098612,False,False,2015-08-07 18:17:43.170000+00:00,1,98799
2,2,259884,0,0,1,0.693147,False,False,2015-08-07 17:56:52.664000+00:00,1,118730
3,2,325215,0,0,3,1.386294,False,False,2015-08-07 18:20:57.845000+00:00,1,148367
4,2,342816,0,0,2,1.098612,False,False,2015-08-07 18:17:24.375000+00:00,1,156464


In [26]:
user_mapping.to_parquet(
    PROCESSED_DIR / "user_mapping.parquet",
    index=False
)

item_mapping.to_parquet(
    PROCESSED_DIR / "item_mapping.parquet",
    index=False
)

encoded_interactions.to_parquet(
    PROCESSED_DIR / "interactions_train_encoded.parquet",
    index=False
)

print("Đã lưu các bảng mapping và tương tác đã mã hóa.")

Đã lưu các bảng mapping và tương tác đã mã hóa.


In [27]:
model_events = events[
    events["split"].isin(["train", "validation", "test"])
].copy()

model_events = model_events.sort_values(
    ["session_id", "event_time", "session_position"]
)

previous_item = (
    model_events.groupby("session_id")["itemid"]
    .shift(1)
)

session_steps = model_events[
    model_events["itemid"].ne(previous_item)
].copy()

session_steps["step"] = (
    session_steps.groupby("session_id")
    .cumcount()
    .astype("int32")
)

session_steps.to_parquet(
    PROCESSED_DIR / "session_events.parquet",
    index=False
)

print("Số hành vi trước khi rút gọn:", len(model_events))
print("Số bước session sau khi rút gọn:", len(session_steps))

Số hành vi trước khi rút gọn: 2755313
Số bước session sau khi rút gọn: 2475048


In [28]:
train_session_steps = session_steps[
    session_steps["split"] == "train"
].copy()

train_session_steps["next_itemid"] = (
    train_session_steps.groupby("session_id")["itemid"]
    .shift(-1)
)

next_item_train = train_session_steps.dropna(
    subset=["next_itemid"]
).copy()

next_item_train["next_itemid"] = (
    next_item_train["next_itemid"].astype("int64")
)

next_item_train = next_item_train[
    [
        "session_id",
        "visitorid",
        "step",
        "itemid",
        "event",
        "next_itemid"
    ]
]

next_item_train.to_parquet(
    SPLIT_DIR / "next_item_train.parquet",
    index=False
)

print("Số cặp next-item dùng huấn luyện:", len(next_item_train))
next_item_train.head()

Số cặp next-item dùng huấn luyện: 583094


,session_id,visitorid,step,itemid,event,next_itemid
4,3,2,0,325215,view,259884
6,3,2,1,259884,view,216305
7,3,2,2,216305,view,342816
8,3,2,3,342816,view,216305
10,3,2,4,216305,view,325215


In [29]:
def create_session_evaluation(data, split_name, train_users, train_items):
    part = data[data["split"] == split_name].copy()

    part["sequence_length"] = (
        part.groupby("session_id")["itemid"]
        .transform("size")
    )

    # Session phải có ít nhất hai sản phẩm
    part = part[part["sequence_length"] >= 2].copy()

    # Sản phẩm cuối là mục tiêu
    part["target_itemid"] = (
        part.groupby("session_id")["itemid"]
        .transform("last")
    )

    # Chỉ giữ lịch sử trước sản phẩm cuối
    history = part[
        part["step"] < part["sequence_length"] - 1
    ].copy()

    history["cold_user"] = (
        ~history["visitorid"].isin(train_users)
    )

    history["cold_target_item"] = (
        ~history["target_itemid"].isin(train_items)
    )

    return history[
        [
            "session_id",
            "visitorid",
            "step",
            "itemid",
            "event",
            "target_itemid",
            "cold_user",
            "cold_target_item"
        ]
    ]

validation_eval = create_session_evaluation(
    session_steps,
    "validation",
    train_users,
    train_items
)

test_eval = create_session_evaluation(
    session_steps,
    "test",
    train_users,
    train_items
)

validation_eval.to_parquet(
    SPLIT_DIR / "session_validation.parquet",
    index=False
)

test_eval.to_parquet(
    SPLIT_DIR / "session_test.parquet",
    index=False
)

print("Số dòng lịch sử validation:", len(validation_eval))
print("Số dòng lịch sử test:", len(test_eval))

print("\nCold-start trong validation:")
print(
    validation_eval[
        ["cold_user", "cold_target_item"]
    ].value_counts()
)

Số dòng lịch sử validation: 64243
Số dòng lịch sử test: 66083

Cold-start trong validation:
cold_user  cold_target_item
True       False               46155
False      False               14610
True       True                 2213
False      True                 1265
Name: count, dtype: int64


In [30]:
split_summary = (
    events[events["split"].isin(["train", "validation", "test"])]
    .groupby("split", observed=True)
    .agg(
        number_of_events=("itemid", "size"),
        number_of_users=("visitorid", "nunique"),
        number_of_items=("itemid", "nunique"),
        number_of_sessions=("session_id", "nunique"),
        start_time=("event_time", "min"),
        end_time=("event_time", "max")
    )
    .reset_index()
)

split_summary.to_csv(
    REPORT_DIR / "data_split_summary.csv",
    index=False
)

split_summary

,split,number_of_events,number_of_users,number_of_items,number_of_sessions,start_time,end_time
0,test,275484,158004,77496,182821,2015-09-02 17:49:22.567000+00:00,2015-09-18 02:59:47.788000+00:00
1,train,2204470,1123755,212915,1396273,2015-05-03 03:00:04.384000+00:00,2015-08-18 04:23:01.129000+00:00
2,validation,275359,156925,77664,182534,2015-08-18 04:23:22.451000+00:00,2015-09-02 17:49:11.563000+00:00


In [31]:
# 1. Không còn sự kiện không hợp lệ
assert events["event"].isin(
    ["view", "addtocart", "transaction"]
).all()

# 2. Các cột quan trọng không bị thiếu
assert events[
    ["timestamp", "visitorid", "event", "itemid", "event_time"]
].isna().sum().sum() == 0

# 3. Mỗi session chỉ thuộc một split
max_splits_per_session = (
    events.groupby("session_id")["split"]
    .nunique()
    .max()
)
assert max_splits_per_session == 1

# 4. Score phải dương
assert (interactions_train["interaction_score"] > 0).all()

# 5. user_idx và item_idx không được thiếu
assert encoded_interactions["user_idx"].isna().sum() == 0
assert encoded_interactions["item_idx"].isna().sum() == 0

# 6. Không dùng boundary session trong dữ liệu mô hình
assert not session_steps["split"].eq("boundary_drop").any()

print("TẤT CẢ KIỂM TRA ĐỀU THÀNH CÔNG.")
print("Dữ liệu đã sẵn sàng để xây dựng mô hình.")

TẤT CẢ KIỂM TRA ĐỀU THÀNH CÔNG.
Dữ liệu đã sẵn sàng để xây dựng mô hình.
